
# Scenario Testing

Definitive smoke/perf harness for the Household Energy ABM. Use it to:
- load and enrich the household GeoJSON with HIDP socio-demographics,
- slice to a ward/LSOA for fast iteration,
- run baseline plus policy variants (income/edu HP grants, top-user HPs, kids vs elderly tweaks, social-rent HP) with quick diagnostics and optional exports.

Time budget: baseline smoke (~2 days simulated) runs in under a minute on a laptop; multi-year sections later are heavier—read their notes before executing.



## How to run (order)
1) Edit paths/toggles in the next code cell (`GEOJSON`, `CLIMATE`, `HIDP_CSV`, `AREA_COLUMN`, `AREA_CODE`, `DAYS`).
2) Run **Load geometry...** and check printed counts/filters.
3) Run **Build and run a short simulation** to confirm agent/person counts and total_energy trend.
4) Run **Quick coverage** to see HIDP/schedule fill rates.
5) Pick scenarios to test (each is self-contained):
   - HP grants (income/education, 24h) — fastest.
   - Multi-year HP split plot — heaviest; prefer ward-only.
   - Top-user HPs — medium (1y baseline + 2y policy).
   - Kids vs elderly policies — fast (1 week window).
   - Social-rent HP — fast (24h).
6) (Optional) Save parquet outputs for downstream analysis.

Runtime tips:
- Keep agent-level collection for short runs; disable it for multi-year sweeps.
- Narrow to a ward/LSOA (`AREA_CODE`) to cut runtime drastically.


## Prerequisites
- Environment: `esa_mesa` venv with requirements installed.
- Data: GeoJSON (`epc_abm_newcastle.geojson`), climate parquet, HIDP CSV (`hidp_uprn_matches_tiered.csv`).
- Set `AREA_COLUMN`/`AREA_CODE` to a ward/LSOA you care about (e.g., `ward_code` = `E05011456`).



## Scope and levers

- HIDP merge adds `hh_n_people`, income band, tenure, schedule_type, education; resident counts follow `hh_n_people` with caps from config.
- Schedules respect `schedule_type` with jitter; `occupancy_count` is exported for plotting/animation.
- Bedroom-aware scaling via `households.bedroom_multiplier` in `config_defaults.yaml`.
- Config overrides are small YAMLs that deep-merge onto defaults—use the pattern shown below.
- Animation helpers (`make_animation.py`) color all households and black-out when empty.

Default paths assume the repo layout; adjust in the next cell before running.


## How to use custom configs
- Defaults live in `household_energy/config_defaults.yaml`.
- Create a minimal override YAML with only the keys you need; it deep-merges onto defaults.
- Example:
```yaml
meta:
  name: my_experiment
  date: 2026-01-26
model:
  heating_setpoint_C: 18.0
  heatpump_adoption_rate: 0.4
households:
  hidp_csv: data/hidp_uprn_matches_tiered.csv
  merge_on: uprn_chr
  geojson_uprn_field: UPRN
schedules:
  wfh_share: 0.30
```
- Pass via `config_path="my_experiment.yaml"` when constructing `EnergyModel`, or `--config-path` on the CLI runner.
- Common knobs: `model` (setpoints, slopes, spikes, HP adoption), `households` (join keys, resident caps, bedroom multipliers), `schedules` (profiles, WFH share), `envelope_levers`, `systems`.


In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

from household_energy.model import EnergyModel
from household_energy.run import _safe_sum_household_energy
from household_energy.config import load_config

# Optional helper from scripts/enrichment_check.py if you want coverage stats later



## Paths and toggles
Edit the next cell before running anything else; `OUTDIR` is created automatically. Set `USE_ENRICHMENT = False` to skip the HIDP CSV.


In [ ]:
# User settings
GEOJSON = Path("../data/epc_abm_newcastle.geojson")
CLIMATE = Path("../data/ncc_2t_timeseries_2010_2039.parquet")
HIDP_CSV = Path("../data/hidp_uprn_matches_tiered.csv")
OUTDIR = Path("results/scenario")
OUTDIR.mkdir(parents=True, exist_ok=True)

USE_ENRICHMENT = True  # set False to skip the CSV merge
AREA_COLUMN = "ward_code"  # e.g., "lsoa_code", "ward_code", or None
AREA_CODE = "E05011456"    # code to filter; set "" to skip filtering
DAYS = 2                   # keep runs short for notebook


## Load geometry, optional HIDP enrichment, and filter area

In [ ]:
gdf = gpd.read_file(GEOJSON)
print(f"Loaded {len(gdf):,} rows from {GEOJSON}")

if USE_ENRICHMENT:
    hidp_df = pd.read_csv(HIDP_CSV, low_memory=False)
    hidp_df.columns = [c.strip() for c in hidp_df.columns]
    gdf['UPRN'] = gdf['UPRN'].astype(str).str.strip()
    hidp_df['uprn_chr'] = hidp_df['uprn_chr'].astype(str).str.strip()
    before = len(gdf)
    gdf = gdf.merge(hidp_df, how="left", left_on="UPRN", right_on="uprn_chr")
    unmatched = gdf['uprn_chr'].isna().sum()
    print(f"Enriched households: {before:,} → {len(gdf):,}; unmatched HIDP rows: {unmatched:,}")
else:
    print("Skipping HIDP enrichment (no-enrichment mode).")

if AREA_CODE:
    col = AREA_COLUMN or next((c for c in gdf.columns if 'lsoa' in c.lower() and 'code' in c.lower()), None)
    if col and col in gdf.columns:
        mask = gdf[col].astype(str).str.upper() == AREA_CODE.upper()
        gdf = gdf.loc[mask].copy()
        print(f"Filtered to {len(gdf):,} rows on {col} == {AREA_CODE}")
    else:
        print("Area column not found; using full dataset.")


## Build and run a short simulation
This uses the updated schedules (from `schedule_type` when present) and household-size driven resident counts. Agent-level collection is enabled for inspection.

In [ ]:
model = EnergyModel(
    gdf=gdf,
    climate_parquet=str(CLIMATE),
    climate_start=None,
    collect_agent_level=True,
    agent_collect_every=1,
)
print(f"Households: {len(model.household_agents):,} | Persons: {len(model.person_agents):,}")

steps = DAYS * 24
for h in range(steps):
    model.step()
print(f"Ran {steps} hours")


## Quick coverage / enrichment stats

In [ ]:

agent_df = model.agent_dc.get_agent_vars_dataframe()
hh = agent_df[agent_df['agent_type'] == 'household'].copy()

cols = [
    'hidp','hh_n_people','hh_children','hh_income_band','hh_edu_detail',
    'dwelling_bucket','tenure','size_band','schedule_type','schedule_profile','occupancy_count'
]
for c in cols:
    if c in hh.columns:
        pct = 100 * hh[c].notna().mean()
        print(f"{c:16s}: {pct:5.1f}% non-null")

print("\nTop schedule_type:")
if 'schedule_type' in hh.columns:
    print(hh['schedule_type'].value_counts(dropna=False).head())

print("\nTop schedule_profile (assigned):")
if 'schedule_profile' in hh.columns:
    print(hh['schedule_profile'].value_counts(dropna=False).head())


## Spot-check energy outputs
Model-level frame for the short run.

In [ ]:
model_df = model.model_dc.get_model_vars_dataframe().copy()
model_df.head()


## (Optional) Save to disk for downstream analysis

In [ ]:

model_parquet = OUTDIR / "model_timeseries.parquet"
agent_parquet = OUTDIR / "agent_timeseries.parquet"

model.model_dc.get_model_vars_dataframe().to_parquet(model_parquet)
model.agent_dc.get_agent_vars_dataframe().to_parquet(agent_parquet)
print(f"Saved model -> {model_parquet}")
print(f"Saved agent -> {agent_parquet}")


## Notes on new levers and schedules
- **Baseline**: fixed 0.40 kWh/h meter anchor with mild area/property-type scaling; structural levers (SAP/envelope/fuel) now affect heating only.
- **HIDP & socio-demographics**: joined by UPRN; non-null coverage logged above.
- **Household size**: `hh_n_people` drives resident counts (capped by config); default=2 if missing.
- **Schedules**: `schedule_type` → archetypes with jitter; exported `schedule_profile` shows the assigned pattern. Occupancy_count is tracked for away/home logic.
- **Targeting**: `hh_income_band`, `tenure`, `dwelling_bucket`, children flag, and education are now in agent exports for policy filters.



## Policy scenario: targeted heat-pump grants for lower-income, medium-education households

We’ll compare a **baseline** run to a **targeted HP grant** run where only households in income quantiles 1–2 and medium education receive priority heat-pump eligibility.

- Target cohort: `hh_income_band` ∈ {`q1_lowest`, `q2_low`} AND `hh_edu_detail` ∈ {`education_other`, `upper_further`}.
- Intervention: mark these homes as heat-pump candidates (`is_heatpump_candidate=1`, `heatpump_candidate_class='priority'`) and set `heatpump_adoption_rate=1.0` in a short override config.
- Window: 24 hours (keep quick).
- Outputs: baseline kWh, policy kWh, ΔkWh printed; negative Δ means savings for the targeted group.


In [ ]:
import tempfile, yaml

def run_scenario(gdf_in, config_override, label):
    m = EnergyModel(
        gdf=gdf_in,
        climate_parquet=str(CLIMATE),
        climate_start=None,
        collect_agent_level=False,
        agent_collect_every=24,
        config_path=config_override,
    )
    hours = 24
    for _ in range(hours):
        m.step()
    total_kwh = float(m.model_dc.get_model_vars_dataframe()["total_energy"].sum())
    return total_kwh

# Baseline copy
gdf_base = gdf.copy()

# Targeted HP cohort
mask_target = (
    gdf["hh_income_band"].isin(["q1_lowest", "q2_low"]) &
    gdf["hh_edu_detail"].isin(["education_other", "upper_further"])
)
print(f"Target cohort size: {mask_target.sum():,} households")

gdf_hp = gdf.copy()
# Make only the target cohort eligible & priority
gdf_hp["is_heatpump_candidate"] = 0
gdf_hp.loc[mask_target, "is_heatpump_candidate"] = 1
gdf_hp.loc[mask_target, "heatpump_candidate_class"] = "priority"

# Write a tiny override config with 100% adoption of eligible homes
with tempfile.NamedTemporaryFile("w", suffix=".yaml", delete=False) as tmp:
    override_cfg_path = tmp.name
    yaml.safe_dump({
        "meta": {"name": "hp_target_lowincome", "notes": "HP grants to q1/q2 medium edu"},
        "model": {"heatpump_adoption_rate": 1.0}
    }, tmp)

print(f"Using override config: {override_cfg_path}")

baseline_kwh = run_scenario(gdf_base, None, "baseline")
policy_kwh   = run_scenario(gdf_hp, override_cfg_path, "hp_grant")

delta = policy_kwh - baseline_kwh
print(f"Baseline total kWh (24h): {baseline_kwh:,.0f}")
print(f"Policy total kWh   (24h): {policy_kwh:,.0f}")
print(f"ΔkWh (policy - base): {delta:,.0f} kWh")


### Interpretation
- Negative ΔkWh means the targeted HP grants lowered total load (expected if heating demand dominates).
- Cohort size gives you a sense of how many homes were affected; adjust the targeting sets or adoption rate as needed.
- You can extend this pattern to other levers (e.g., smart meters, envelope flags) by setting the corresponding columns before the scenario run.



## Multi-year scenario: HP grants vs baseline (ward-level)

We’ll run a longer window (default 5 years) for a single ward and compare annual household energy between homes **with** vs **without** heat pumps under the targeted grant. To keep runtime reasonable, agent-level collection is disabled; we accumulate annual kWh on the fly.

Params:
- `RUN_YEARS`: set to 5 for full test (can lower for quick checks)
- `START_UTC`: aligns to climate timestamps
- Area filter reused from earlier cells (ward_code by default)

Outputs: a printed annual table (`annual_df`) and a plot of GWh with vs without heat pumps. Interpretation: a lower "with HP" curve shows savings from the grant over time; flat or rising gaps may signal saturation or rebound.


In [ ]:
import numpy as np
import pandas as pd

RUN_YEARS = 5
START_UTC = "2020-01-01T00:00:00Z"

# Build policy cohort (reuse mask_target from above if present; else create new)
try:
    mask_target
except NameError:
    mask_target = (
        gdf["hh_income_band"].isin(["q1_lowest", "q2_low"]) &
        gdf["hh_edu_detail"].isin(["education_other", "upper_further"])
    )
    print(f"Target cohort size (constructed): {mask_target.sum():,}")

gdf_long = gdf.copy()
gdf_long["is_heatpump_candidate"] = 0
gdf_long.loc[mask_target, "is_heatpump_candidate"] = 1
gdf_long.loc[mask_target, "heatpump_candidate_class"] = "priority"

model_long = EnergyModel(
    gdf=gdf_long,
    climate_parquet=str(CLIMATE),
    climate_start=START_UTC,
    collect_agent_level=False,
    agent_collect_every=168,
    config_path=None,
)

hh_list = model_long.household_agents
n_hh = len(hh_list)
hh_ids = [h.unique_id for h in hh_list]
hh_has_hp = np.array([getattr(h, "has_heatpump", False) for h in hh_list], dtype=bool)

hours = RUN_YEARS * 365 * 24
print(f"Running {hours:,} hours (~{RUN_YEARS} years) for {n_hh:,} households …")

annual_energy = {}
start_ts = pd.Timestamp(START_UTC, tz="UTC")

for step in range(1, hours + 1):
    model_long.step()
    ts = start_ts + pd.Timedelta(hours=step-1)
    year = ts.year
    if year not in annual_energy:
        annual_energy[year] = np.zeros(n_hh, dtype=float)
    # accumulate current-hour energy per household
    for i, h in enumerate(hh_list):
        annual_energy[year][i] += getattr(h, "energy_consumption", 0.0)
    if step % (365*24) == 0:
        print(f" … completed year {year}")

rows = []
for year, arr in annual_energy.items():
    rows.append({
        "year": year,
        "with_hp_kwh": float(arr[hh_has_hp].sum()),
        "without_hp_kwh": float(arr[~hh_has_hp].sum()),
        "with_hp_avg": float(arr[hh_has_hp].mean()) if hh_has_hp.any() else np.nan,
        "without_hp_avg": float(arr[~hh_has_hp].mean()) if (~hh_has_hp).any() else np.nan,
    })
annual_df = pd.DataFrame(rows).sort_values("year")
annual_df


### Plot: annual energy with vs without heat pumps

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7,4))
ax.plot(annual_df['year'], annual_df['with_hp_kwh']/1e6, marker='o', label='With HP (total, GWh)')
ax.plot(annual_df['year'], annual_df['without_hp_kwh']/1e6, marker='o', label='Without HP (total, GWh)')
ax.set_ylabel('Annual household energy (GWh)')
ax.set_xlabel('Year')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


### Notes
- This run is compute-heavy (hours = years × 365 × 24). Reduce `RUN_YEARS` if needed.
- Accumulation is per-household per hour; with ~4k homes and 5 years this is manageable but may take a few minutes.
- Targeting is tied to the income/education cohort defined above; adjust `mask_target` to test other levers.
- `has_heatpump` is read from the HouseholdAgent after policy adoption; baseline status is embedded in the same run, so the split reflects which homes received HPs.



## Scenario: Heat pumps for highest energy consumers (income-agnostic)

This scenario targets the top X% of household energy users (by baseline annual kWh) and makes them heat-pump candidates with full adoption. Income/education are ignored for targeting.

Parameters:
- `TOP_FRAC`: fraction of households to treat (e.g., 0.1 = top 10%)
- `RUN_YEARS_SHORT`: quick multi-year window (default 2) for speed

Outputs: baseline kWh vs policy kWh over `RUN_YEARS_SHORT` and ΔkWh (policy - base). Interpretation: negative Δ indicates savings among high users; adjust `TOP_FRAC` to trade off cohort size vs impact.


In [ ]:
TOP_FRAC = 0.10
RUN_YEARS_SHORT = 2

# 1) Short baseline run to rank households by energy
rank_model = EnergyModel(
    gdf=gdf.copy(),
    climate_parquet=str(CLIMATE),
    climate_start=START_UTC,
    collect_agent_level=True,
    agent_collect_every=24,
)
rank_hours = 365 * 24
for _ in range(rank_hours):
    rank_model.step()

agent_df_rank = rank_model.agent_dc.get_agent_vars_dataframe()
baseline_hh = agent_df_rank[agent_df_rank['agent_type']=='household']
annual_energy = baseline_hh.groupby('AgentID')['energy_consumption'].sum().sort_values(ascending=False)
cut = int(len(annual_energy) * TOP_FRAC)
cut = max(cut, 1)
target_ids = set(annual_energy.head(cut).index)
print(f"Selected top {TOP_FRAC:.0%} => {len(target_ids):,} households")

# 2) Build policy GDF: mark these as HP priority
policy_gdf = gdf.copy()
policy_gdf['is_heatpump_candidate'] = policy_gdf['UPRN'].astype(str).isin(target_ids).astype(int)
policy_gdf.loc[policy_gdf['is_heatpump_candidate']==1, 'heatpump_candidate_class'] = 'priority'

# 3) Run policy vs baseline over shorter multi-year window
hours_short = RUN_YEARS_SHORT * 365 * 24

def run_total(gdf_in):
    m = EnergyModel(
        gdf=gdf_in,
        climate_parquet=str(CLIMATE),
        climate_start=START_UTC,
        collect_agent_level=False,
        agent_collect_every=168,
    )
    for _ in range(hours_short):
        m.step()
    return float(m.model_dc.get_model_vars_dataframe()['total_energy'].sum())

base_total = run_total(gdf)
policy_total = run_total(policy_gdf)
print(f"Baseline kWh over {RUN_YEARS_SHORT}y: {base_total:,.0f}")
print(f"Policy   kWh over {RUN_YEARS_SHORT}y: {policy_total:,.0f}")
print(f"ΔkWh (policy - base): {policy_total - base_total:,.0f}")


### Notes
- This uses a one-year baseline to rank households, then applies HPs for a shorter multi-year comparison.
- Adjust `TOP_FRAC` and `RUN_YEARS_SHORT` to trade off runtime vs signal.
- Ranking uses household energy consumption from the baseline run (AgentID matches UPRN after the merge).


## Scenario: policy targeting households with children (school-run) vs elderly (retired)

We’ll run two policy tweaks over a short window and compare against baseline:
- **Kids-focused**: apply loft + glazing upgrades to households with `hh_children=True` or `schedule_type` in {`family_with_children`, `single_parent_with_children`, `school_run` profile assigned}.
- **Elderly-focused**: apply smart meters and a small heating setback (–1°C) to `schedule_type` = `retired_household`.

Outputs: ΔkWh vs baseline for each policy.


In [ ]:
import numpy as np
import pandas as pd

POLICY_HOURS = 7 * 24  # one week for speed

# Baseline copy
gdf_base = gdf.copy()

# Identify cohorts
kids_mask = (
    gdf.get("hh_children", pd.Series(False, index=gdf.index)).fillna(False).astype(bool)
) | gdf.get("schedule_type", pd.Series("", index=gdf.index)).str.contains("children|school_run", case=False, na=False)

elderly_mask = gdf.get("schedule_type", pd.Series("", index=gdf.index)).str.contains("retired", case=False, na=False)

print(f"Kids cohort: {kids_mask.sum():,} | Elderly cohort: {elderly_mask.sum():,}")

# Kids policy: envelope upgrades
kids_gdf = gdf.copy()
kids_gdf.loc[kids_mask, "loft_ins_flag"] = 1
kids_gdf.loc[kids_mask, "glazing_flag"] = 1

# Elderly policy: smart meters + heating setback via config override
elderly_gdf = gdf.copy()
elderly_gdf.loc[elderly_mask, "meter_type"] = "smart"

import tempfile, yaml
with tempfile.NamedTemporaryFile("w", suffix=".yaml", delete=False) as tmp:
    elderly_cfg = tmp.name
    yaml.safe_dump({
        "meta": {"name": "elderly_setback", "notes": "retired hh smart + setback"},
        "model": {"heating_setpoint_C": float(load_config().model.get("heating_setpoint_C", 18.5)) - 1.0}
    }, tmp)
print(f"Elderly override config: {elderly_cfg}")

def run_total_kwh(gdf_in, cfg=None):
    m = EnergyModel(
        gdf=gdf_in,
        climate_parquet=str(CLIMATE),
        climate_start=START_UTC,
        collect_agent_level=False,
        agent_collect_every=24,
        config_path=cfg,
    )
    for _ in range(POLICY_HOURS):
        m.step()
    return float(m.model_dc.get_model_vars_dataframe()["total_energy"].sum())

base_kwh = run_total_kwh(gdf_base)
kids_kwh = run_total_kwh(kids_gdf)
elderly_kwh = run_total_kwh(elderly_gdf, cfg=elderly_cfg)

print(f"Baseline (kWh): {base_kwh:,.0f}")
print(f"Kids policy ΔkWh: {kids_kwh - base_kwh:,.0f}")
print(f"Elderly policy ΔkWh: {elderly_kwh - base_kwh:,.0f}")



### Quick cohort energy comparison
Outputs:
- ΔkWh vs baseline for kids-policy and elderly-policy runs (printed in the previous code cell).
- Per-household average kWh over the policy week for kids vs non-kids, elderly vs non-elderly (this cell).
Interpretation:
- Negative ΔkWh or lower per-household averages indicate the policy reduced consumption.
- Compare cohort averages to see if savings are concentrated in the targeted group.


In [ ]:

# Per-household averages from baseline one-week run
m_check = EnergyModel(
    gdf=gdf_base,
    climate_parquet=str(CLIMATE),
    climate_start=START_UTC,
    collect_agent_level=True,
    agent_collect_every=24,
)
for _ in range(POLICY_HOURS):
    m_check.step()

a_df = m_check.agent_dc.get_agent_vars_dataframe()
hh_df = a_df[a_df['agent_type']=='household'].copy()
week_energy = hh_df.groupby('AgentID')['energy_consumption'].sum()

# Align masks to AgentID index for safe boolean indexing
kids_mask_idx = kids_mask.reindex(week_energy.index, fill_value=False)
elderly_mask_idx = elderly_mask.reindex(week_energy.index, fill_value=False)

kids_avg = week_energy[kids_mask_idx].mean() if kids_mask_idx.any() else float('nan')
nokids_avg = week_energy[~kids_mask_idx].mean() if (~kids_mask_idx).any() else float('nan')
elderly_avg = week_energy[elderly_mask_idx].mean() if elderly_mask_idx.any() else float('nan')
nonelderly_avg = week_energy[~elderly_mask_idx].mean() if (~elderly_mask_idx).any() else float('nan')

print(f"Kids avg kWh/week: {kids_avg:,.2f} | Non-kids: {nokids_avg:,.2f}")
print(f"Elderly avg kWh/week: {elderly_avg:,.2f} | Others: {nonelderly_avg:,.2f}")



## Diagnostics: ward time-series and property-type breakdown
Short ward run with total energy over time and a property-type aggregate. Useful as a smoke check after changing configs or data filters.


In [ ]:

import matplotlib.pyplot as plt

DIAG_DAYS = 3
WARD_FOR_DIAG = AREA_CODE  # reuse current filter

# Filter to ward if available
diag_gdf = gdf.copy()
if WARD_FOR_DIAG:
    col = AREA_COLUMN or next((c for c in diag_gdf.columns if 'lsoa' in c.lower() and 'code' in c.lower()), None)
    if col and col in diag_gdf.columns:
        mask = diag_gdf[col].astype(str).str.upper() == WARD_FOR_DIAG.upper()
        diag_gdf = diag_gdf.loc[mask].copy()
        print(f"Diagnostics: {len(diag_gdf):,} dwellings in {col}={WARD_FOR_DIAG}")
    else:
        print("Diagnostics: area column not found, using full gdf")

diag_model = EnergyModel(
    gdf=diag_gdf,
    climate_parquet=str(CLIMATE),
    climate_start=START_UTC,
    collect_agent_level=True,
    agent_collect_every=24,
)
diag_hours = DIAG_DAYS * 24
for _ in range(diag_hours):
    diag_model.step()

diag_model_df = diag_model.model_dc.get_model_vars_dataframe().iloc[1:].reset_index(drop=True)
diag_model_df['hour'] = diag_model_df.index

plt.figure(figsize=(8,4))
plt.plot(diag_model_df['hour'], diag_model_df['total_energy'])
plt.xlabel('Hour')
plt.ylabel('Total energy (kWh)')
plt.title('Total energy over time (diagnostic check)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Property type breakdown (if columns present)
prop_cols = [c for c in diag_model_df.columns if c in diag_model.energy_by_type.keys()]
if prop_cols:
    diag_model_df[prop_cols].sum().sort_values(ascending=False).plot(kind='bar', figsize=(7,4))
    plt.ylabel('kWh over run')
    plt.title('Energy by property type (diagnostic check)')
    plt.tight_layout()
    plt.show()



These plots mirror the quick diagnostics in this notebook: a short ward run with total energy vs time and a property-type aggregate. Adjust `DIAG_DAYS` or `WARD_FOR_DIAG` as needed.



## Scenario quick reference (what to look for)
- Baseline smoke: agent/person counts and total_energy trend sanity.
- HP grants (income/edu): ΔkWh over 24h; note the printed cohort size.
- Multi-year HP split: annual GWh with vs without HPs for the targeted cohort.
- Top-user HPs: kWh over `RUN_YEARS_SHORT` for treated top fraction vs baseline.
- Kids vs Elderly: ΔkWh for each policy plus per-household week averages.
- Social-rent HP: 24h ΔkWh for `tenure == social_rent`.

Copy/paste masks (income, tenure, dwelling_bucket, children) to craft new scenarios—most cells expect the same `gdf` columns.


## Notes on performance and reproducibility
- **Agent-level data** is heavy; disable when doing multi-year sweeps.
- **Seeds**: we seed RNGs in the code cells for reproducible schedules and heat-pump assignment; if you change seeds, results will vary.
- **Color scaling**: when using the animation script, you can pass `--color-max` and `--all-households` for consistent legends; black-out uses occupancy where available.
- **Config overrides**: keep them minimal and dated; see the README section on configs.



## Scenario: Heat pumps for social rent households (council savings estimate)
We target all `tenure == social_rent`, force HP eligibility, and compare 24h kWh vs baseline.
Outputs: baseline kWh, policy kWh, and ΔkWh (policy - baseline). Negative Δ indicates savings for the treated stock.


In [ ]:
# Social rent HP scenario
import tempfile, yaml

def run_scenario(gdf_in, config_override=None):
    m = EnergyModel(
        gdf=gdf_in,
        climate_parquet=str(CLIMATE),
        climate_start=None,
        collect_agent_level=False,
        agent_collect_every=24,
        config_path=config_override,
    )
    hours = 24
    for _ in range(hours):
        m.step()
    return float(m.model_dc.get_model_vars_dataframe()["total_energy"].sum())

# Baseline
gdf_base = gdf.copy()

# Policy: mark social rent as HP candidates
mask_social = gdf["tenure"].astype(str).str.lower() == "social_rent"
print(f"Social rent households: {mask_social.sum():,}")

gdf_hp = gdf.copy()
gdf_hp["is_heatpump_candidate"] = 0
gdf_hp.loc[mask_social, "is_heatpump_candidate"] = 1
gdf_hp.loc[mask_social, "heatpump_candidate_class"] = "priority"

with tempfile.NamedTemporaryFile("w", suffix=".yaml", delete=False) as tmp:
    hp_social_cfg = tmp.name
    yaml.safe_dump({
        "meta": {"name": "hp_social_rent", "notes": "HP grants to social rent"},
        "model": {"heatpump_adoption_rate": 1.0}
    }, tmp)

baseline_kwh = run_scenario(gdf_base, None)
policy_kwh   = run_scenario(gdf_hp, hp_social_cfg)

print(f"Baseline total kWh (24h): {baseline_kwh:,.0f}")
print(f"Policy   total kWh (24h): {policy_kwh:,.0f}")
print(f"ΔkWh (policy - base): {policy_kwh - baseline_kwh:,.0f} kWh")
